# Настройка бустинга и категориальные признаки

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Собирать небольшой табличный бустинг с подготовкой категорий и выбирать число этапов без теста.

## Настраиваем всё решение

Глубина задаёт сложность одного дерева, число этапов — сложность суммы, learning_rate — размер добавления. Менять только один параметр и игнорировать остальные недостаточно. Начинают с простого baseline и небольшой заранее заданной сетки вариантов.

Категории нельзя всегда передавать произвольными числами: модель может принять код за порядок. В этой практике используем знакомый OneHotEncoder и GradientBoostingRegressor в Pipeline. Это конкретный доступный способ; разные библиотеки бустинга могут поддерживать категории иначе. Новую зависимость ради другого пакета не вводим.

## Разобранный пример: остановка по validation

Пусть после 10,20,30 этапов validation MSE равно 5,4,6. Сохраняем модель на 20 этапах, а не последнюю на 30. Это **ранняя остановка**: прекращаем усложнение при отсутствии улучшения по заранее выбранному правилу. Обучающая ошибка при этом может продолжать уменьшаться.

В ноутбуке GridSearchCV сравнивает 5 и 10 этапов на трёх разбиениях перекрёстной проверки. В каждом разбиении преобразования обучаются только на обучающей части; scoring='neg_mean_squared_error' возвращает отрицательную MSE, потому что поисковая процедура максимизирует показатель. Поэтому более высокий score означает меньшую ошибку, а не «отрицательную физическую ошибку».

После выбора итоговый тест открывают один раз. При маленькой таблице итоговая оценка груба; особенно полезно посмотреть ошибки на неизвестных категориях и крайних значениях числовых признаков. Протокол важнее высокой цифры на синтетических данных.

`remainder="passthrough"` оставляет остальные столбцы после выбранного кодирования. `best_params_` хранит выбранные настройки, `best_score_` — средний показатель внутренней проверки лучшего варианта.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold
X = pd.DataFrame({"size": np.arange(18), "paper": ["А", "Б"] * 9})
y = np.arange(18, dtype=float) * 2
prepare = ColumnTransformer([("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["paper"])], remainder="passthrough")
model = Pipeline([("prepare", prepare), ("boost", GradientBoostingRegressor(max_depth=1, random_state=17))])
search = GridSearchCV(model, {"boost__n_estimators": [5, 10]}, cv=KFold(3, shuffle=True, random_state=17), scoring="neg_mean_squared_error")
search.fit(X, y)
assert search.best_params_["boost__n_estimators"] in [5, 10]
assert search.best_score_ <= 0
print(search.best_params_, "CV MSE:", -search.best_score_)


## Типичные ошибки

Не сообщайте отрицательный score как отрицательную MSE. Кодирование категорий нужно заново обучать на обучающей части каждого разбиения.

## Задачи

### 1. Базовая

Что означает learning_rate?

<details>
<summary>Подсказка</summary>

Это множитель новой поправки.

</details>



### 2. Базовая

Почему GridSearchCV использует отрицательную MSE?

<details>
<summary>Подсказка</summary>

Он выбирает максимум score.

</details>



### 3. Прикладная

Для ошибок 5,4,6 при 10,20,30 этапах какой вариант выбрать?

<details>
<summary>Подсказка</summary>

Минимум внутренней ошибки.

</details>



### 4. Прикладная

Какой score лучше: −3 или −7?

<details>
<summary>Подсказка</summary>

Поменяйте знак обратно.

</details>



### 5. Олимпиадная

Можно ли выбирать число этапов по итоговому тесту?

<details>
<summary>Подсказка</summary>

Остановка — тоже подбор.

</details>



### 6. Олимпиадная
У трёх объектов категории А, Б, В и ответы 1,0,1. Дерево с одним порогом проверяет «код ≤ t» и выдаёт постоянный ответ в каждой ветке. Сравните кодировки А→1, Б→2, В→3 и А→1, Б→3, В→2: в какой можно без ошибок разделить ответы одним порогом? Что это говорит о произвольной нумерации категорий?

<details>
<summary>Подсказка</summary>

Выпишите ответы в порядке возрастания кодов и проверьте места возможного порога.

</details>



## Итоги и самопроверка

Объясните совместную роль глубины, темпа и числа этапов. Где хранится лучший вариант и почему он может быть не последним?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Темп добавления каждого этапа к суммарному прогнозу.

### Задача 2

Минус превращает минимизацию ошибки в максимизацию показателя.

### Задача 3

20 этапов по этому протоколу.

### Задача 4

−3 соответствует меньшей MSE=3.

### Задача 5

Нет. Для неё нужна внутренняя validation или CV.

### Задача 6

В первой кодировке порядок ответов 1,0,1. Любой порог между соседними кодами оставит разные ответы хотя бы в одной ветке; порог вне диапазона тоже не поможет. Во второй кодировке порядок 1,1,0: порог t=2.5 отделяет А и В от Б без ошибок. Одна и та же задача стала проще для дерева только из-за произвольного порядка кодов. Для неупорядоченных категорий следует использовать подходящее кодирование или специальную обработку категорий, а не приписывать номерам предметный порядок.